``EmbeddingPreprocessor.fetch_embeddings`` downloads a protein language model (ESM-2, ESM-1b, ProtT5, ProstT5) from the Hugging Face Hub and computes its embeddings: one mean/max/cls-pooled vector per protein (``mode='protein'``) or a per-residue ``(L, D)`` array per protein (``mode='residue'``). It requires the ``embed`` extra (``pip install 'aaanalysis[embed]'``); the heavy dependencies are imported lazily, so the rest of the class works without them.

In [1]:
import numpy as np
import aaanalysis as aa
aa.options["verbose"] = False

# A small dataset; the smallest model keeps this fast (weights downloaded once).
df_seq = aa.load_dataset(name="DOM_GSEC", n=2)

embp = aa.EmbeddingPreprocessor()
# Per-protein embeddings: one mean-pooled vector per protein.
X, df_out = embp.fetch_embeddings(df_seq, mode="protein", model="esm2_t6_8M",
                                pooling="mean", return_df=True)
print("per-protein matrix:", X.shape)
aa.display_df(df_out, n_rows=10, show_shape=True)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

per-protein matrix: (4, 320)
DataFrame shape: (4, 9)


,entry,sequence,label,tmd_start,tmd_stop,jmd_n,tmd,jmd_c,embeddings_ok
1,Q14802,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,0,37,59,NSPFYYDWHS,LQVGGLICAGVLCAMGIIIVMSA,KCKCKFGQKS,True
2,Q86UE4,MAARSWQDELAQQAE...SPKQIKKKKKARRET,0,50,72,LGLEPKRYPG,WVILVGTGALGLLLLFLLGYGWA,AACAGARKKR,True
3,P05067,MLPGLALLLLAAWTA...GYENPTYKFFEQMQN,1,701,723,FAEDVGSNKG,AIIGLMVGGVVIATVIVITLVML,KKKQYTSIHH,True
4,P14925,MAGRARSGLLLLLLG...EEEYSAPLPKPAPSS,1,868,890,KLSTEPGSGV,SVVLITTLLVIPVLVLLAIVMFI,RWKKSRAFGD,True


Per-residue embeddings are raw, unbounded floats; normalize them with :meth:`~EmbeddingPreprocessor.encode` before :meth:`CPP.run_num`. The per-protein matrix above feeds :meth:`AAclust.select_proteins` or a classifier directly.

In [2]:
# Per-residue embeddings: one (L, D) array per protein.
emb = embp.fetch_embeddings(df_seq, mode="residue", model="esm2_t6_8M")
for entry, arr in emb.items():
    print(entry, "->", arr.shape)

# Normalize to [0, 1] for CPP.run_num.
dict_num = embp.encode(df_seq=df_seq, embeddings=emb)
print("encoded entries:", len(dict_num))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Q14802 -> (87, 320)
Q86UE4 -> (582, 320)
P05067 -> (770, 320)
P14925 -> (976, 320)
encoded entries: 4


Key parameters: ``model`` selects the PLM (``esm2_t6_8M`` … ``esm2_t33_650M``, ``prott5_xl_u50``, ``prostt5``); ``pooling`` is ``'mean'`` / ``'max'`` / ``'cls'`` (``'cls'`` only for models with a leading token, i.e. ESM not ProtT5); ``device`` is ``'auto'`` / ``'cpu'`` / ``'cuda'`` / ``'mps'``; ``batch_size`` and ``max_length`` control throughput and truncation; ``on_failure`` (``'nan'`` / ``'drop'`` / ``'raise'``) handles per-entry failures; and ``allow_oversized`` bypasses the hardware-size warning. The :meth:`~EmbeddingPreprocessor.pool_embeddings` helper pools per-residue arrays explicitly.

In [3]:
# Max-pool the per-residue arrays into a per-protein matrix.
X_max = embp.pool_embeddings(emb, pooling="max", df_seq=df_seq)
print("max-pooled matrix:", X_max.shape)

max-pooled matrix: (4, 320)
